# 5.3. Forward Propagation, Backward Propagation, and Computational Graphs
D2L의 Forward Propagation, Backward Propagation, and Computational Graphs장을 PyTorch 기준으로 정리함.

순전파 (Forward Propagation)
입력 -> 예측 -> 손실 계산

역전파 (Backpropagation)
손실 -> gradient 계산 -> 가중치 수정

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 순전파

순전파(Forward Propagation)는 입력 데이터를 신경망에 넣고 입력층에서 출력층 방향으로 계산을 진행하는 과정이다.

하나의 은닉층을 가진 MLP를 생각해보면

```text
입력 x 
  ↓ 
Linear 
  ↓
z 
  ↓ 
Activation 
  ↓ 
h 
  ↓ 
Linear 
  ↓ 
출력 o 
  ↓ 
Loss 
  ↓ 
L
```

먼저 입력 x가 첫 번째 가중치 W1을 통과한다.

$$
z = W^{(1)}x
$$

z는 아직 활성화 함수를 통과하지 않은 값이다. 그리고 활성화 함수를 적용한다.

$$
h = \phi(z)
$$

h는 은닉층의 최종 출력이다.

이 값을 다시 출력층에 전달한다.

$$
o = W^{(2)}h
$$

그리고 실제 정답 y와 비교하여 손실을 계산한다.

$$
L = l(o, y)
$$

결국 순전파의 목적은
현재 가중치로 예측을 만들고 -> 예측이 얼마나 틀렸는지 계산하는 것 이다.

## 2. 계산 그래프

신경망의 계산 과정을 그림처럼 나타낸 것을 계산 그래프(Computational Graph)라고 한다.

앞에서 계산한 MLP를 단순하게 표현하면 다음과 같다.

    x - W1 -> z - φ(활성함수) -> h - W2 -> o -> 정답 y와 비교 -> L

각 변수들은 서로를 의존하고 있다.  
h를 계산하려면 z가 필요하고, o를 계산하려면 h가 필요하고...

그래서 x -> z -> h -> o -> L  
순서대로 계산해야 한다.

이게 순전파 방향이다.

계산 그래프를 이해하는 것이 중요한 이유는 역전파가 이 그래프를 반대 방향으로 이동해서 이다.

## 3. 역전파

역전파(Backpropagation)는 손실 함수가 각 가중치에 얼마나 영향을 받았는지 계산하는 과정이다.

순전파가 입력 -> 출력 방향이라면
역전파는 출력 -> 입력 방향이다.

역전파에서 우리가 알고 싶은 것은 결국 다음과 같은 값이다.

$$
\frac{\partial L}{\partial W}
$$

가중치 W를 조금 바꾸면 Loss가 얼마나 변하는가?

를 계산하는 것이다.

이 값이 gradient(기울기)이다.

## 4. 연쇄법칙과 역전파

문제는 Loss와 앞쪽 가중치가 직접 연결되어 있지 않다.

예를 들어서

    W1 → z → h → o → L

이것 처럼 여러 계산을 거쳐서 Loss에 영향을 준다.

그래서 W1이 Loss에 얼마나 영향을 주는지 알고 싶으면 중간 계산을 다 따라 가야한다.  
이때 사용하는게 연쇄법칙(Chain Rule)이다.

간단한 예를 생각해보자.

x -> y -> z 일때

$$
y = f(x)
$$

$$
z = g(y)
$$

라고 하자.

x가 z에 얼마나 영향을 주는지 알고 싶다면

$$
\frac{\partial z}{\partial y}
\frac{\partial y}{\partial x}
$$

처럼 중간 관계를 연결하면 된다. 신경망에서도 똑같다.

```text
W¹
 ↓
z
 ↓
h
 ↓
o
 ↓
L
```

Loss에서 시작해서 하나씩 미분값을 전달한다.

```text
L
↓
∂L/∂o
↓
∂L/∂h
↓
∂L/∂z
↓
∂L/∂W¹
```

그래서 gradient가 뒤쪽에서 앞쪽으로 전파된다고 해서 Backpropagation이라고 부른다.

## 5. MLP에서 실제 역전파 흐름

앞에서 사용한 MLP를 보면

    x -> W1 -> z -> Activation -> h -> W2 -> o -> Loss -> L

역전파에서는 반대로 움직인다.

### 1. 출력 gradient 계산

먼저 Loss가 출력 o에 얼마나 영향을 받는지 계산한다.
$$
\frac{\partial L}{\partial o}
$$

    출력값이 조금 바뀌면 Loss가 얼마나 변하는가?

### 2. W2의 gradient 계산

출력층은

$$
o = W^{(2)}h
$$
로 계산 됬다.

Loss에서 전달된 gradient와 h를 이용해

$$
\frac{\partial L}{\partial W^{(2)}}
$$

    W2를 어느 방향으로 수정해야 Loss가 감소하는가?

### 3. 은닉층으로 gradient 전달

gradient를 출력층에서 은닉층으로 보낸다.

L -> o -> h

$$
\frac{\partial L}{\partial h}
$$

이걸 계산한다.

### 4. 활성화 함수도 역으로 진행

순전파에서 

$$
h = \phi(z)
$$

를 계산했기 때문에 역전파에서는 활성화 함수의 미분값이 필요하다.

$$
\frac{\partial L}{\partial h}\odot\phi'(z)
$$

순전파 흐름
z -> Activation -> h

역전파 흐름
gradient -> Activation의 미분 -> gradient

활성화 함수도 gradient가 얼마나 전달될지 결정한다.

### 5. W1의 gradient 계산

마지막으로 첫 번째 가중치까지 gradient를 전달한다.

$$
\frac{\partial L}{\partial W^{(1)}}
$$

이제 두 가중치 모두 gradient가 있다.

Optimizer는 이 gradient를 이용하여 가중치를 수정한다.

## 6. 순전파와 역전파를 한 번에 보기

전체 학습 과정을 연결하면 다음과 같다.

### 순전파

```text
입력 x
  ↓
W¹
  ↓
z
  ↓
Activation
  ↓
h
  ↓
W²
  ↓
출력 o
  ↓
Loss
  ↓
L
```

### 역전파

```text
L
  ↓
출력의 gradient
  ↓
W² gradient 계산
  ↓
은닉층으로 gradient 전달
  ↓
Activation 미분
  ↓
W¹ gradient 계산
```

### 가중치 수정

W¹ 업데이트
W² 업데이트

그리고 다시 새로운 순전파가 시작된다.

```text
순전파
   ↓
Loss 계산
   ↓
역전파
   ↓
Gradient 계산
   ↓
가중치 업데이트
   ↓
순전파
   ↓
...
```

이 과정을 반복하면서 Loss를 줄여나가는 것이 신경망 학습이다.

## 7. Weight Decay가 있다면

D2L에서는 설명을 위해서 Weight Decay도 사용한다.

일반 Loss를 L이라고 하고

Weight Decay에 의한 규제항을 s라고 하면 

최종 목적함수는 J = L + s 가 된다.

모델은 예측 오차를 줄이는 것 외에도 가중치가 지나치게 커지는 것도 억제하도록 학습한다.

그래서 역전파엣는 단순 Loss뿐 아니라 Weight Decay가 가중치에 주는 영향도 gradient에 포함된다.

## 8. 왜 순전파의 값을 저장하나

역전파를 자세히 보면 순전파에서 계산했던 값들이 다시 사용된다.

예를 들어서 W2의 gradient를 계산하려면 은닉층 출력 h가 필요하다.

할성화 함수를 미분할 때도 순전파에서 계산했던 z같은 값이 필요할 수도 있다.

```text
순전파하면서 중간값 계산 
    ↓ 
중간값 저장 
    ↓ 
Loss 계산 
    ↓ 
역전파하면서 저장한 값 재사용
```

그래서 일반적으로 학습은 추론보다 더 많은 메모리를 사용한다고 한다.

특히 네트워크가 깊어질수록, Batch Size가 커질수록 저장해야할 중간값이 증가한다.

## 9. 오늘의 정리

- 순전파(Forward Propagation)​는 입력에서 출력 방향으로 계산하며 예측값과 Loss를 구하는 과정이다.
- 계산 그래프(Computational Graph)​는 신경망의 변수와 연산 사이의 관계를 표현한다.
- 역전파(Backpropagation)​는 계산 그래프를 반대 방향으로 이동하면서 각 파라미터의 gradient를 계산한다.
- 역전파의 핵심 원리는 미적분의 연쇄법칙(Chain Rule)​이다.
- Gradient는 가중치를 조금 바꾸었을 때 Loss가 얼마나 변하는가를 의미한다.
- 역전파는 순전파에서 계산한 중간값들을 다시 사용한다.
- 따라서 학습 시에는 중간값을 저장해야 하므로 추론보다 일반적으로 더 많은 메모리가 필요하다.
- 전체 신경망 학습은 순전파 → Loss 계산 → 역전파 → 가중치 업데이트를 반복하는 과정이다.